# 4강: 문서 텍스트 추출 실습

이 노트북에서는 다양한 문서 형식(PDF, DOCX, HWP 등)에서
텍스트를 추출하는 방법을 실습합니다.

## 4.1 파일 형식별 추출 전략

| 형식 | 라이브러리 | 내부 구조 |
|------|------------|----------|
| PDF  | PyMuPDF (fitz) | 바이너리, 페이지 단위 텍스트 레이어 |
| DOCX | python-docx | ZIP + XML (OOXML) |
| XLSX | openpyxl | ZIP + XML (OOXML), 셀 단위 |
| PPTX | python-pptx | ZIP + XML (OOXML), 슬라이드 단위 |
| HWPX | zipfile + xml | ZIP + XML (Open HWPX) |
| HWP  | olefile | OLE2 (Compound File), 바이너리 태그 레코드 |
| TXT  | 내장 open() | UTF-8/CP949/EUC-KR/Latin-1 순 디코딩 |

## 4.2 텍스트 파일 다중 인코딩 디코딩

In [ ]:
def extract_plain(raw_bytes: bytes) -> str | None:
    """바이트 데이터를 UTF-8 → CP949 → EUC-KR → Latin-1 순으로 디코딩 시도"""
    for enc in ('utf-8', 'cp949', 'euc-kr', 'latin-1'):
        try:
            return raw_bytes.decode(enc)
        except (UnicodeDecodeError, UnicodeError):
            continue
    return None

# UTF-8 텍스트
utf8_text = '안녕하세요 Python!'.encode('utf-8')
print(f'UTF-8: {extract_plain(utf8_text)}')

# CP949 (한글 Windows 기본 인코딩) 텍스트
cp949_text = '안녕하세요 Python!'.encode('cp949')
print(f'CP949: {extract_plain(cp949_text)}')

# Latin-1 (최후의 수단 — 모든 바이트를 수용)
latin1_text = bytes(range(128, 256))
result = extract_plain(latin1_text)
print(f'Latin-1 폴백: 성공={result is not None}, 길이={len(result) if result else 0}')

## 4.3 DOCX (Open XML) 구조 탐색

In [ ]:
import zipfile
import io

# DOCX는 실제로 ZIP 파일입니다
# word/document.xml에 본문이 XML로 저장되어 있습니다

# DOCX 구조 예시 (실제 파일 없이 구조만 설명)
docx_structure = {
    '[Content_Types].xml': 'MIME 타입 매핑',
    '_rels/.rels': '패키지 관계 정의',
    'word/document.xml': '★ 본문 텍스트 (w:p > w:r > w:t)',
    'word/styles.xml': '스타일 정의',
    'word/fontTable.xml': '폰트 테이블',
    'word/media/': '이미지 등 미디어 파일',
}

print('DOCX(ZIP) 내부 구조:')
for path, desc in docx_structure.items():
    print(f'  {path:35s} — {desc}')

print()
print('python-docx 추출 코드:')
print('''
from docx import Document
doc = Document('example.docx')
texts = [p.text for p in doc.paragraphs if p.text.strip()]
full_text = '\\n'.join(texts)
''')

## 4.4 HWP 바이너리 태그 레코드 파싱

HWP5는 OLE2(Compound File Binary Format) 컨테이너 안에
BodyText 스트림이 태그 레코드 형태로 저장됩니다.

### 태그 레코드 헤더 구조 (4바이트)
```
[tag_id: 10비트] [level: 10비트] [size: 12비트]
```
- tag_id == 67 → HWPTAG_PARA_TEXT (문단 텍스트)
- size == 0xFFF → 확장 크기 (다음 4바이트가 실제 크기)

In [ ]:
# HWP 태그 레코드 파싱 시뮬레이션

HWP_TAG_MASK     = 0x3FF
HWP_SIZE_MASK    = 0xFFF
HWP_SIZE_SHIFT   = 20
HWPTAG_PARA_TEXT = 67

# 인라인 제어 코드 (비텍스트 요소)
INLINE_CTRL_CODES = frozenset({1, 2, 3, 11, 12, 14, 15, 16, 17, 18, 21, 22, 23})
INLINE_CTRL_EXTRA = 12  # 제어 문자 뒤 추가 바이트

def parse_tag_header(header: int) -> tuple[int, int]:
    """4바이트 태그 헤더에서 tag_id와 size를 추출"""
    tag_id = header & HWP_TAG_MASK
    size = (header >> HWP_SIZE_SHIFT) & HWP_SIZE_MASK
    return tag_id, size

def decode_para_text(chunk: bytes) -> str:
    """HWP PARA_TEXT 청크를 UTF-16LE로 디코딩 (제어 코드 필터링)"""
    chars = []
    j = 0
    while j < len(chunk) - 1:
        code = int.from_bytes(chunk[j:j+2], 'little')
        j += 2
        if code >= 0x0020:        # 인쇄 가능 문자
            chars.append(chr(code))
        elif code in (0x0D, 0x0A): # 줄바꿈
            chars.append('\n')
        elif code == 0x09:         # 탭
            chars.append(' ')
        elif code in INLINE_CTRL_CODES:
            j += INLINE_CTRL_EXTRA  # 제어 코드 보조 데이터 건너뛰기
    return ''.join(chars)

# 테스트: 가짜 PARA_TEXT 데이터
test_text = '안녕하세요 HWP!'
test_chunk = test_text.encode('utf-16-le')
print(f'원본: {test_text}')
print(f'UTF-16LE 바이트: {test_chunk.hex()}')
print(f'디코딩: {decode_para_text(test_chunk)}')

# 태그 헤더 파싱 테스트
sample_header = (HWPTAG_PARA_TEXT) | (0 << 10) | (len(test_chunk) << 20)
tag_id, size = parse_tag_header(sample_header)
print(f'\n태그 헤더: 0x{sample_header:08X}')
print(f'tag_id={tag_id} (HWPTAG_PARA_TEXT={HWPTAG_PARA_TEXT}), size={size}')

## 4.5 HWP zlib 3단계 해제 전략

In [ ]:
import zlib

original = b'Hello World! ' * 100  # 테스트 데이터

# 3가지 압축 방식
raw_deflate = zlib.compress(original, level=6)     # zlib 래퍼 포함
# raw deflate (wbits=-15) — HWP5 표준
compressor = zlib.compressobj(level=6, wbits=-15)
raw_only = compressor.compress(original) + compressor.flush()

print(f'원본 크기:          {len(original)} 바이트')
print(f'raw deflate 크기:   {len(raw_only)} 바이트 (wbits=-15, HWP5 표준)')
print(f'zlib wrapped 크기:  {len(raw_deflate)} 바이트 (wbits=15, 비표준 HWP)')

def hwp_decompress(data: bytes) -> bytes | None:
    """SeekSeek의 3단계 해제 전략 시뮬레이션"""
    # 1차: raw deflate (HWP5 표준)
    try:
        return zlib.decompress(data, -15)
    except zlib.error:
        pass
    # 2차: zlib wrapped (비표준)
    try:
        return zlib.decompress(data, 15)
    except zlib.error:
        pass
    # 3차: gzip 자동 감지
    try:
        return zlib.decompress(data, 47)
    except zlib.error:
        return None

# raw deflate 해제
result1 = hwp_decompress(raw_only)
print(f'\nraw deflate 해제: {"성공" if result1 == original else "실패"}')

# zlib wrapped 해제 (1차 실패 → 2차 성공)
result2 = hwp_decompress(raw_deflate)
print(f'zlib wrapped 해제: {"성공" if result2 == original else "실패"}')

## 4.6 PPTX ZIP 폴백 추출

In [ ]:
import re

# PPTX도 ZIP 파일. 슬라이드 XML에서 <a:t> 태그로 텍스트 추출
PPTX_T_RE = re.compile(r'<a:t[^>]*>(.*?)</a:t>', re.DOTALL)

sample_xml = '''
<p:sp>
  <p:txBody>
    <a:p>
      <a:r>
        <a:t>SeekSeek 소개</a:t>
      </a:r>
    </a:p>
    <a:p>
      <a:r>
        <a:t>NTFS MFT 기반 초고속 검색</a:t>
      </a:r>
    </a:p>
  </p:txBody>
</p:sp>
'''

texts = [m.group(1).strip() for m in PPTX_T_RE.finditer(sample_xml) if m.group(1).strip()]
print('PPTX <a:t> 추출 결과:')
for t in texts:
    print(f'  {t}')

## 4.7 SeekSeek의 extract_text() 사용하기

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from core.extractor import extract_text

# 현재 프로젝트의 Python 파일 텍스트 추출
test_file = os.path.abspath('../config.py')
if os.path.exists(test_file):
    text = extract_text(test_file)
    if text:
        print(f'파일: {test_file}')
        print(f'추출된 텍스트 길이: {len(text):,}자')
        print(f'첫 200자:')
        print(text[:200])
    else:
        print('텍스트 추출 실패')
else:
    print(f'테스트 파일 없음: {test_file}')

## 요약

- **텍스트 파일**: UTF-8 → CP949 → EUC-KR → Latin-1 순 디코딩
- **DOCX/XLSX/PPTX**: ZIP + XML (OOXML) 구조, 라이브러리 또는 직접 파싱
- **HWPX**: ZIP + XML, `Contents/*.xml` 내 `<t>` 태그
- **HWP5**: OLE2 바이너리, BodyText 스트림의 태그 레코드(HWPTAG_PARA_TEXT=67) 파싱
- **HWP 압축**: 3단계 zlib 해제 (raw → wrapped → gzip)
- **SeekSeek**: `extract_text()` 하나로 확장자별 자동 라우팅